# Project 15 — CineNoord Cinema Ticket Sales (Bronze → Silver → Gold + unit test)

A take-home style exercise on a messy POS export from a small Dutch cinema chain.
Finance needs revenue they can trust, and the team lead asked for at least one **tested function**.

**What this notebook shows**

- Reading everything as string first, then cleaning, then casting — never the other way round
- Measuring the damage **before** deciding what to drop (a `> 0` filter also deletes nulls)
- Repairing a broken column instead of deleting the row when the money on that row is still valid
- A cross-field data-quality check (`booking_code` vs `city`) that is **flagged**, not silently fixed
- Gold questions in SQL *and* the DataFrame API, including a window function
- A pytest unit test with a **hand-written** expected result, including a deliberate tie

## 1 · Read everything as string

The source is a CSV from an old POS system, so I don't let Spark guess types.
Reading as string means nothing is silently turned into `null` at the door — I decide what is broken, and when.

In [ ]:
cinema_tickets_raw_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .load("/Volumes/dev/spark_db/datasets/mini-projects/raw_data/cinema_tickets_15.csv")
)

cinema_tickets_raw_df.display()

## 2 · Profile before touching anything

Row count, column count and the schema. This is the baseline every later number is compared against: **123 rows**.

In [ ]:
cinema_tickets_raw_df.printSchema()
print(cinema_tickets_raw_df.count())          # 123 rows
print(len(cinema_tickets_raw_df.columns))     # 10 columns

## 3 · Normalise the text first

Trim everywhere, `initcap` for names and cities, `lower` for the ticket type.
Doing this **before** deduplication matters: two rows that differ only by `" amsterdam "` vs `"Amsterdam"` are not duplicates until the text is normalised.

In [ ]:
from pyspark.sql.functions import trim, lower, initcap, col

cinema_tickets_normalized_df = cinema_tickets_raw_df.withColumns({
    "ticket_id":      trim(col("ticket_id")),
    "booking_code":   trim(col("booking_code")),
    "sale_date":      trim(col("sale_date")),
    "city":           trim(initcap(col("city"))),
    "movie_title":    trim(initcap(col("movie_title"))),
    "ticket_type":    trim(lower(col("ticket_type"))),
    "seats":          trim(col("seats")),
    "price_total":    trim(col("price_total")),
    "customer_first": trim(initcap(col("customer_first"))),
    "customer_last":  trim(initcap(col("customer_last"))),
})

cinema_tickets_normalized_df.display()

## 4 · Clean the money text before the cast

`price_total` arrives as `€ 45,50`, `45.50 €`, `€45.50`. The currency sign goes, the decimal comma becomes a dot —
and only then does the column become a number. Casting first would throw the value away.

In [ ]:
from pyspark.sql.functions import lit, replace

cinema_tickets_normalized_df = cinema_tickets_normalized_df.withColumn(
    "price_total",
    replace(
        replace(
            replace(
                replace(col("price_total"), lit("€ "), lit("")),
                lit(" €"), lit("")),
            lit(","), lit(".")),
        lit("€"), lit(""))
)

cinema_tickets_normalized_df.display()

## 5 · Look at the actual values

`groupBy().count()` on every column is the cheapest way to see what is really in there.
This is where the junk shows up: `unknown` ticket types, `N/A` seats, negative seats, `ERROR` prices.

> Note the call is `.display()` on its own — wrapping it in `print()` returns `None`, which is a mistake I have made before.

In [ ]:
for c in ["city", "movie_title", "ticket_type", "seats", "price_total"]:
    cinema_tickets_normalized_df.groupBy(c).count().orderBy(col("count").desc()).display()

## 6 · Placeholders become real nulls

`unknown`, `n/a` and `ERROR` are not values, they are missing data wearing a costume.
Turning them into `null` first means every later count (`isNull`) tells the truth.

In [ ]:
from pyspark.sql.functions import when

placeholders = ["unknown", "n/a", "error"]

cinema_tickets_normalized_df = cinema_tickets_normalized_df.withColumns({
    c: when(lower(col(c)).isin(placeholders), None).otherwise(col(c))
    for c in cinema_tickets_normalized_df.columns
})

cinema_tickets_normalized_df.display()

## 7 · Types, name and city code

Four different date formats live in this file, so `coalesce(try_to_date × 4)` handles them and I check that nothing is left unparsed.

Two things the team lead asked for:

- **`full_name`** — built with `concat_ws(" ", ...)` *after* the parts were trimmed and capitalised
- **`city_code`** — the first three letters of `booking_code`

In [ ]:
from pyspark.sql.functions import try_to_date, coalesce, concat_ws, substring

cinema_tickets_typed_df = cinema_tickets_normalized_df.withColumns({
    "sale_date": coalesce(
        try_to_date("sale_date", "d/M/yyyy"),
        try_to_date("sale_date", "yyyy-M-d"),
        try_to_date("sale_date", "MMM d, yyyy"),
        try_to_date("sale_date", "MMMM d, yyyy"),
    ),
    "seats":       col("seats").cast("int"),
    "price_total": col("price_total").cast("decimal(10,2)"),
    "full_name":   concat_ws(" ", col("customer_first"), col("customer_last")),
    "city_code":   substring(col("booking_code"), 1, 3),
})

# every date parsed?
print(cinema_tickets_typed_df.filter(col("sale_date").isNull()).count())   # 0
cinema_tickets_typed_df.display()

## 8 · Measure the damage before deleting anything

This is the step that decides how honest the revenue number will be.

A filter like `price_total > 0` looks harmless, but a `null` compared with anything is `null`, so
**that filter deletes the unknown rows too** — quietly. Before removing anything I count what is actually there
and I check how much money sits on the rows I would be throwing away.

In [ ]:
from pyspark.sql.functions import sum as _sum

print("null price :", cinema_tickets_typed_df.filter(col("price_total").isNull()).count())   # 5
print("null seats :", cinema_tickets_typed_df.filter(col("seats").isNull()).count())         # 1
print("seats <= 0 :", cinema_tickets_typed_df.filter(col("seats") <= 0).count())             # 2

# how much revenue sits on the rows with a broken seat count?
(cinema_tickets_typed_df
    .filter(col("seats").isNull() | (col("seats") <= 0))
    .select("ticket_id", "city", "seats", "price_total")
    .display())

print("revenue at risk:",
      cinema_tickets_typed_df.filter(col("seats").isNull() | (col("seats") <= 0))
      .agg(_sum("price_total")).collect()[0][0])       # 200.00

## 9 · Repair the column, keep the row

Three rows have an impossible seat count (`-1`, `-2`, `N/A`) — but their price is perfectly valid, and together they
carry **€200 of real revenue**. Deleting those rows would quietly shrink the revenue report to protect a column
that finance is not even asking about.

So: the broken value becomes `null`, the row stays. Same for the five rows whose price could not be recovered —
they stay as an upstream data-quality signal, and `sum()` skips nulls anyway.

In [ ]:
cinema_tickets_typed_df = cinema_tickets_typed_df.withColumn(
    "seats",
    when(col("seats") <= 0, None).otherwise(col("seats"))
)

print(cinema_tickets_typed_df.count())                                        # still 123
print(cinema_tickets_typed_df.filter(col("seats").isNull()).count())          # 3
print(cinema_tickets_typed_df.filter(col("seats") <= 0).count())              # 0

## 10 · Duplicates: prove it, then drop it

`ticket_id` is supposed to be the key, so `count` vs `count(distinct)` is the test.
Three ids appear twice — and when I look at those rows, **every column** is identical, so they are true duplicates
(a double export), not two different sales sharing an id. That is what makes a full-row `dropDuplicates()` the right call.

In [ ]:
print(cinema_tickets_typed_df.count())                              # 123
print(cinema_tickets_typed_df.select("ticket_id").distinct().count())  # 120

cinema_tickets_typed_df.groupBy("ticket_id").count().filter(col("count") > 1).display()
cinema_tickets_typed_df.filter(col("ticket_id").isin("TKT-1034", "TKT-1079", "TKT-1082")).display()

In [ ]:
cinema_tickets_clean_df = cinema_tickets_typed_df.dropDuplicates()

print(cinema_tickets_clean_df.count())                                   # 120
print(cinema_tickets_clean_df.select("ticket_id").distinct().count())    # 120  -> one row per ticket

## 11 · Cross-field check: does `booking_code` agree with `city`?

The booking code starts with a city code (`AMS`, `ROT`, `UTR`). That gives me a free consistency check between two
independent fields — the kind of check that catches upstream bugs.

I use an explicit mapping instead of comparing the first three letters of the city name. Comparing letters happens to
work for these three cities, but it is luck: a fourth city would break it silently.

In [ ]:
from pyspark.sql.functions import create_map, upper
from itertools import chain

city_to_code = {"Amsterdam": "AMS", "Rotterdam": "ROT", "Utrecht": "UTR"}
mapping = create_map([lit(x) for x in chain(*city_to_code.items())])

cinema_tickets_clean_df = cinema_tickets_clean_df.withColumn(
    "city_code_mismatch",
    (upper(col("city_code")) != mapping[col("city")])
)

cinema_tickets_clean_df.filter(col("city_code_mismatch")).select(
    "ticket_id", "booking_code", "city", "city_code"
).display()

print(cinema_tickets_clean_df.filter(col("city_code_mismatch")).count())   # 2

**What I found:** two rows disagree — `TKT-1041` has an `AMS` booking code but says *Rotterdam*, and `TKT-1095` has a
`ROT` code but says *Utrecht*.

**What I did:** I flagged them and kept them. I have no way of knowing which of the two fields is wrong — the booking
system could have issued the code in the wrong city, or the city field could have been typed by hand. Overwriting one
with the other would invent data and destroy the evidence. Two flagged rows out of 120 do not move the revenue report,
but they are exactly what the POS team needs to see.

## 12 · Write the silver table (idempotent) and check it

`mode("overwrite")` means re-running the notebook produces the same table, never a doubled one.
Then three final checks: row count, grain, and where the nulls are.

In [ ]:
cinema_tickets_clean_df.write.mode("overwrite").saveAsTable("cinema_tickets_silver")

silver_df = spark.read.table("cinema_tickets_silver")

print("rows          :", silver_df.count())                                  # 120
print("distinct ids  :", silver_df.select("ticket_id").distinct().count())   # 120  -> grain: one row per ticket
print("null price    :", silver_df.filter(col("price_total").isNull()).count())   # 5
print("null seats    :", silver_df.filter(col("seats").isNull()).count())         # 3
print("flagged rows  :", silver_df.filter(col("city_code_mismatch")).count())     # 2

# Gold questions

**G1.** Total revenue per city — which city earns the most?
**G2.** Monthly revenue (`yyyy-MM`) — which month was the strongest?
**G3.** Top 2 movies per city by revenue — window function required.

Each one is written twice, in SQL and with the DataFrame API, to keep both languages fresh.

## G1 · Revenue per city

No rows are excluded. The five rows with an unknown price contribute nothing to a `SUM` anyway, and the three rows
with a repaired seat count keep their (valid) revenue.

In [ ]:
%sql
select
    city,
    sum(price_total) as total_revenue
from cinema_tickets_silver
group by city
order by total_revenue desc

-- Utrecht 2402.75  >  Rotterdam 1921.25  >  Amsterdam 1267.25

In [ ]:
from pyspark.sql.functions import sum as _sum

revenue_per_city_df = (
    silver_df.groupBy("city")
    .agg(_sum("price_total").alias("total_revenue"))
    .orderBy(col("total_revenue").desc())
)

revenue_per_city_df.display()

## G2 · Monthly revenue

`date_format(sale_date, "yyyy-MM")` — not `month()` and not `monthname()`.
A month name on its own merges the same month across different years; the file happens to cover one year here,
but the habit is what protects the next report.

In [ ]:
%sql
with monthly as (
    select
        date_format(sale_date, 'yyyy-MM') as sale_month,
        price_total
    from cinema_tickets_silver
)

select
    sale_month,
    sum(price_total) as monthly_revenue
from monthly
group by sale_month
order by monthly_revenue desc

-- strongest month: 2025-05 with 1285.50

In [ ]:
from pyspark.sql.functions import date_format

monthly_revenue_df = (
    silver_df
    .withColumn("sale_month", date_format("sale_date", "yyyy-MM"))
    .groupBy("sale_month")
    .agg(_sum("price_total").alias("monthly_revenue"))
    .orderBy(col("monthly_revenue").desc())
)

monthly_revenue_df.display()

## G3 · Top 2 movies per city — window function

Two steps, and the order matters: **aggregate first** (revenue per city × movie), **then** rank inside each city.
Ranking raw ticket rows would rank single sales, not movies.

I picked `dense_rank()` on purpose. Two movies in Amsterdam can end up with exactly the same revenue, and in that case
I would rather the report showed both than silently picked one alphabetically. The trade-off is honest: with a tie,
"top 2" can return three rows, so anything consuming this table must accept that. If the contract required exactly two
rows per city, I would use `row_number()` with an explicit tie-breaker (`movie_title asc`) — a decision, not a default.

In [ ]:
%sql
with per_movie as (
    select
        city,
        movie_title,
        sum(price_total) as total_revenue
    from cinema_tickets_silver
    group by city, movie_title
),

ranked as (
    select
        city,
        movie_title,
        total_revenue,
        dense_rank() over (partition by city order by total_revenue desc) as rn
    from per_movie
)

select * from ranked
where rn <= 2
order by city, rn

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank

w = Window.partitionBy("city").orderBy(col("total_revenue").desc())

top_2_movies_per_city_df = (
    silver_df.groupBy("city", "movie_title")
    .agg(_sum("price_total").alias("total_revenue"))
    .withColumn("rn", dense_rank().over(w))
    .filter(col("rn") <= 2)
    .orderBy("city", "rn")
)

top_2_movies_per_city_df.display()

# Unit test

The G3 logic is the part most likely to break silently, so it moves out of the notebook into a module and gets a test.

Three rules for the function: it **returns** a DataFrame (no `display`), it does **not** create its own Spark session,
and it does **not** read any file. That is what makes it testable.

`cinema_functions.py`:

```python
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank, col, sum


def top_movies_per_city(df, n=2):
    df = df.groupBy("city", "movie_title").agg(
        sum("price_total").alias("total_revenue")
    )
    w = Window.partitionBy("city").orderBy(col("total_revenue").desc())
    df = df.withColumn("rn", dense_rank().over(w))
    df = df.filter(col("rn") <= n)
    return df
```

## The test input has a tie on purpose

Five hand-made rows. Zwolle has **Bravo and Charlie both on 300** — that is the whole point of the test:
it pins down what my ranking choice does in a tie.

The expected result is written **by hand**, from reading the table. Generating it by running the function first would
be circular: wrong code would approve its own output.

With `dense_rank()` the expectation is four rows — Alpha (1), Bravo (2), Charlie (2) and Echo (1).
With `row_number()` it would be three, and Charlie would silently disappear. Same data, different report,
and the test is what makes that choice visible.

`test_cinema_functions.py`:

```python
import pytest
from pyspark.sql import SparkSession
from pyspark.testing import assertDataFrameEqual
from cinema_functions import top_movies_per_city


@pytest.fixture(scope="session")
def spark():
    return SparkSession.builder.getOrCreate()


def test_top_movies_per_city(spark):
    input_df = spark.createDataFrame(
        [
            ("Zwolle", "Alpha",   500.0),
            ("Zwolle", "Bravo",   300.0),
            ("Zwolle", "Charlie", 300.0),
            ("Zwolle", "Delta",   100.0),
            ("Assen",  "Echo",     90.0),
        ],
        ["city", "movie_title", "price_total"],
    )

    expected_df = spark.createDataFrame(
        [
            ("Zwolle", "Alpha",   500.0, 1),
            ("Zwolle", "Bravo",   300.0, 2),
            ("Zwolle", "Charlie", 300.0, 2),
            ("Assen",  "Echo",     90.0, 1),
        ],
        "city string, movie_title string, total_revenue double, rn int",
    )

    result_df = top_movies_per_city(input_df)
    assertDataFrameEqual(result_df, expected_df)
```

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from cinema_functions import top_movies_per_city

top_movies_per_city(spark.read.table("cinema_tickets_silver")).display()

In [ ]:
import pytest
import sys
sys.dont_write_bytecode = True

retcode = pytest.main(["-v", "test_cinema_functions.py"])
assert retcode == 0        # a non-zero exit code is what CI reads as "failed"

# test_cinema_functions.py::test_top_movies_per_city PASSED   [100%]

# Decision log

| # | Decision | Why |
|---|---|---|
| 1 | Read every column as string, cast at the end | Casting a dirty string either throws or nulls the value; cleaning first keeps the data |
| 2 | `unknown` / `n/a` / `ERROR` → `null` | They are missing values in disguise; as text they poison every count and cast |
| 3 | Kept the 5 rows whose price could not be recovered | `SUM` ignores nulls, so they cost nothing — and they are evidence for the POS team |
| 4 | Repaired impossible seat counts (`-1`, `-2`, `N/A`) instead of deleting the rows | Those rows carry **€200 of valid revenue**; deleting them to protect a column finance doesn't use would understate the report |
| 5 | Normalised the text **before** deduplicating | `" amsterdam "` and `"Amsterdam"` are the same city; dedup can only see that after cleaning |
| 6 | Full-row `dropDuplicates()` (123 → 120) | The three repeated `ticket_id`s were identical in **every** column — a double export, not two sales |
| 7 | Flagged the 2 `booking_code` / `city` disagreements, did not fix them | I cannot know which field is wrong; overwriting one would invent data and hide the problem |
| 8 | `date_format(..., 'yyyy-MM')` for the monthly report | A month name alone merges the same month across years |
| 9 | `dense_rank()` for "top 2 per city" | A tie should show both movies; the cost is that a tie can return three rows, which the test makes explicit |
| 10 | `mode("overwrite")` on the silver table | Re-running the notebook must not duplicate the table |

# Defense questions

**Q1 — Your `city_code` check found disagreements. Why is flagging safer than fixing?**

Because a fix requires knowing which field is right, and I don't. Either the booking system issued the code in the
wrong city or somebody typed the city by hand. If I overwrite one field with the other, the report looks consistent
and the real bug becomes invisible — and the evidence is gone. I flag, keep the row, and report it. To actually fix it
I would need a third source: the venue the screening ran in, or the POS terminal id.

**Q2 — `rank()` vs `row_number()` vs `dense_rank()` on a tie, and why does the test care?**

With two movies on the same revenue, `row_number()` gives 1 and 2 in an arbitrary order — one of them silently
disappears from a "top 2". `rank()` gives 1, 1, 3 and `dense_rank()` gives 1, 1, 2, so both keep the tied pair, and
`rn <= 2` then returns three rows for that city. My hand-written expectation encodes that choice, so if somebody
switches the function to `row_number()` for a "cleaner" result, the test fails and forces the conversation.

**Q3 — Dropping the broken rows vs keeping them with a null?**

I keep them. A `SUM` skips nulls, so a null price costs the revenue report nothing, while a deleted row costs it the
whole sale. That difference was measurable here: the three rows with a broken seat count carried €200. The rule I work
by is "repair the column, not the row" — and only drop a row when the row itself is meaningless, not when one field is.

**Q4 — Your test has 5 rows and the table has 120. What does it actually prove?**

It proves the *logic*: aggregate before ranking, and this specific behaviour on a tie. It proves nothing about today's
data — whether the file arrived complete, whether prices are plausible, whether a city is missing. That is a data
quality check on the pipeline, not a unit test. Unit tests protect me from my own future edits; DQ checks protect me
from the source.

# Key takeaways

- **A `> 0` filter is not a cleaning step, it's a deletion.** `null > 0` is `null`, so the unknown rows go too —
  measure first, then decide.
- **Repair the column, keep the row.** One broken field is not a reason to throw away a valid sale;
  here it was worth €200 out of €5,591.
- **Prove the duplicate before dropping it.** Identical `ticket_id` is a suspicion; identical *every column* is evidence.
- **A cross-field check is free data quality.** `booking_code` and `city` are independent, so they can be compared —
  and when they disagree, flag rather than guess.
- **The ranking function is a business decision.** `dense_rank` shows ties, `row_number` guarantees N rows;
  the unit test is where that decision is written down.